# Notebook 4: Generación de Imágenes (Image-to-Image)

**Autores:** Javier Arroyo | Julia Cano | Paula Durá  
**Asignatura:** Procesamiento de Imágenes  

---

## Objetivo

Generar **nuevas imágenes sintéticas** para cada categoría del dataset (Animales, Ciudad, Comida, Naturaleza, Playa) mediante dos enfoques:

| # | Modelo | Enfoque | Resolución |
|---|--------|---------|-----------|
| 1 | cDCGAN (from scratch) | GAN condicional entrenada desde cero | 64×64 |
| 2 | Stable Diffusion (preentrenado) | Modelo de difusión con prompts | 512×512 |

### Evaluación
A diferencia de clasificación o detección, en generación de imágenes **no existe una métrica numérica única**. La evaluación es primordialmente **cualitativa**: realismo, coherencia con la categoría, diversidad y nivel de detalle.

> **Nota:** El dataset original (`dataset/`) se usa para entrenar la GAN.

---
## 4.1 Configuración

Importamos TensorFlow/Keras para la GAN y `diffusers` (Hugging Face) para Stable Diffusion.

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import torch
from diffusers import StableDiffusionPipeline

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

In [ ]:
DATA_DIR = Path("dataset")
classes = sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()])
print("Clases:", classes)

---
## 4.2 Modelo from scratch: cDCGAN (Conditional Deep Convolutional GAN)

### Fundamento teórico

Una **GAN** (*Generative Adversarial Network*) consiste en dos redes que se entrenan en competición:
- **Generator (G):** Transforma ruido aleatorio $z \sim \mathcal{N}(0, 1)$ en imágenes sintéticas
- **Discriminator (D):** Distingue imágenes reales de las generadas

El entrenamiento sigue un juego minimax:
$$\min_G \max_D \; \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$

### Nuestra variante: **cDCGAN**
- **Conditional:** Ambas redes reciben la etiqueta de clase como entrada, permitiendo generar imágenes de una categoría específica
- **Deep Convolutional:** Usa convoluciones transpuestas (G) y convoluciones con stride (D) en lugar de capas densas

### Expectativas realistas
Con solo ~150 imágenes por clase y resolución 64×64, esperamos resultados limitados (texturas/colores característicos de cada clase), pero no fotorrealismo.

### 4.2.1 Carga y preprocesamiento de datos

Las imágenes se normalizan al rango $[-1, 1]$ (necesario para la activación `tanh` del Generator).

In [ ]:
GAN_IMG_SIZE = (64, 64)
GAN_BATCH_SIZE = 32
LATENT_DIM = 128
GAN_EPOCHS = 100

gan_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels="inferred",
    label_mode="int",
    image_size=GAN_IMG_SIZE,
    batch_size=GAN_BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

gan_class_names = gan_ds.class_names
NUM_CLASSES = len(gan_class_names)
print("GAN class_names:", gan_class_names)

def gan_preprocess(imgs, labels):
    imgs = tf.cast(imgs, tf.float32) / 127.5 - 1.0  # [-1,1]
    return imgs, labels

gan_ds = gan_ds.map(gan_preprocess, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

### 4.2.2 *Sanity check*: ejemplos reales (64×64)

Antes de entrenar, visualizamos las imágenes reales redimensionadas a 64×64 para establecer el "objetivo visual" que la GAN intenta replicar.

In [ ]:
def show_real_gan_batch(dataset, n=10):
    imgs, labels = next(iter(dataset))
    imgs = (imgs.numpy() + 1.0) / 2.0  # [0,1]
    plt.figure(figsize=(12, 4))
    for i in range(min(n, imgs.shape[0])):
        plt.subplot(2, 5, i+1)
        plt.imshow(imgs[i])
        plt.title(gan_class_names[int(labels[i])])
        plt.axis("off")
    plt.suptitle("Ejemplos reales del dataset (64x64)")
    plt.tight_layout()
    plt.show()

show_real_gan_batch(gan_ds, n=10)

### 4.2.3 Arquitectura del Generator

Transforma un vector de ruido $z \in \mathbb{R}^{128}$ concatenado con un *embedding* de clase en una imagen 64×64×3:

```
[z(128) + ClassEmb(32)] → Dense(4×4×512) → ConvT(256) → ConvT(128) → ConvT(64) → ConvT(32) → Conv(3, tanh)
```

Cada bloque usa `BatchNormalization + ReLU` para estabilizar el entrenamiento.

In [ ]:
def build_generator(latent_dim, num_classes):
    noise_in = layers.Input(shape=(latent_dim,), name="noise")
    label_in = layers.Input(shape=(), dtype=tf.int32, name="label")

    label_emb = layers.Embedding(num_classes, 32)(label_in)
    label_emb = layers.Flatten()(label_emb)

    x = layers.Concatenate()([noise_in, label_emb])

    x = layers.Dense(4*4*512, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Reshape((4, 4, 512))(x)

    x = layers.Conv2DTranspose(256, 4, strides=2, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2DTranspose(128, 4, strides=2, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2DTranspose(64, 4, strides=2, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2DTranspose(32, 4, strides=2, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    out = layers.Conv2D(3, 3, padding="same", activation="tanh")(x)
    return keras.Model([noise_in, label_in], out, name="Generator")

G = build_generator(LATENT_DIM, NUM_CLASSES)
G.summary()

### 4.2.4 Arquitectura del Discriminator

Red convolucional con stride que clasifica pares (imagen, clase) como reales o falsos:

```
[Image(64×64×3) concat ClassMap(64×64×1)] → Conv64 → Conv128 → Conv256 → Flatten → Dense(1, logit)
```

Usa `LeakyReLU(0.2) + Dropout(0.3)` para estabilidad (evita que el discriminador sea "demasiado bueno" demasiado rápido).

In [ ]:
def build_discriminator(img_size, num_classes):
    img_in = layers.Input(shape=(*img_size, 3), name="image")
    label_in = layers.Input(shape=(), dtype=tf.int32, name="label")

    label_map = layers.Embedding(num_classes, img_size[0] * img_size[1])(label_in)
    label_map = layers.Reshape((img_size[0], img_size[1], 1))(label_map)

    x = layers.Concatenate(axis=-1)([img_in, label_map])

    x = layers.Conv2D(64, 4, strides=2, padding="same")(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(128, 4, strides=2, padding="same")(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(256, 4, strides=2, padding="same")(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Flatten()(x)
    out = layers.Dense(1)(x)  # logit

    return keras.Model([img_in, label_in], out, name="Discriminator")

D = build_discriminator(GAN_IMG_SIZE, NUM_CLASSES)
D.summary()

### 4.2.5 Entrenamiento adversarial

El entrenamiento alterna entre:
1. **Entrenar D:** Maximizar $\log D(x_{real}) + \log(1 - D(G(z)))$
2. **Entrenar G:** Minimizar $\log(1 - D(G(z)))$ (equivalente a maximizar $\log D(G(z))$)

Usamos Adam con `lr=2e-4, β₁=0.5` (hiperparámetros estándar para DCGANs) y entrenamos durante 100 epochs.

In [ ]:
bce = keras.losses.BinaryCrossentropy(from_logits=True)

g_opt = keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.5)
d_opt = keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.5)

@tf.function
def train_step(real_imgs, real_labels):
    batch_size = tf.shape(real_imgs)[0]

    noise = tf.random.normal((batch_size, LATENT_DIM))
    fake_labels = real_labels
    fake_imgs = G([noise, fake_labels], training=True)

    with tf.GradientTape() as d_tape:
        real_logits = D([real_imgs, real_labels], training=True)
        fake_logits = D([fake_imgs, fake_labels], training=True)
        d_loss_real = bce(tf.ones_like(real_logits), real_logits)
        d_loss_fake = bce(tf.zeros_like(fake_logits), fake_logits)
        d_loss = d_loss_real + d_loss_fake

    d_grads = d_tape.gradient(d_loss, D.trainable_variables)
    d_opt.apply_gradients(zip(d_grads, D.trainable_variables))

    noise2 = tf.random.normal((batch_size, LATENT_DIM))
    with tf.GradientTape() as g_tape:
        gen_imgs = G([noise2, fake_labels], training=True)
        gen_logits = D([gen_imgs, fake_labels], training=True)
        g_loss = bce(tf.ones_like(gen_logits), gen_logits)

    g_grads = g_tape.gradient(g_loss, G.trainable_variables)
    g_opt.apply_gradients(zip(g_grads, G.trainable_variables))

    return d_loss, g_loss

In [ ]:
# Monitorización: grid fijo por clase
FIXED_NOISE = tf.random.normal((NUM_CLASSES * 6, LATENT_DIM))

def generate_grid(epoch):
    labels = np.repeat(np.arange(NUM_CLASSES), 6)
    labels = tf.constant(labels, dtype=tf.int32)

    imgs = G([FIXED_NOISE, labels], training=False)
    imgs = (imgs.numpy() + 1.0) / 2.0

    plt.figure(figsize=(12, 2*NUM_CLASSES))
    idx = 0
    for c in range(NUM_CLASSES):
        for j in range(6):
            plt.subplot(NUM_CLASSES, 6, idx+1)
            plt.imshow(imgs[idx])
            plt.axis("off")
            if j == 0:
                plt.ylabel(gan_class_names[c], rotation=0, labelpad=40, va="center")
            idx += 1
    plt.suptitle(f"Epoch {epoch} — muestras generadas por clase")
    plt.tight_layout()
    plt.show()

In [ ]:
# Entrenamiento del cDCGAN
d_hist, g_hist = [], []

for epoch in range(1, GAN_EPOCHS + 1):
    d_losses, g_losses = [], []

    for real_imgs, real_labels in gan_ds:
        d_loss, g_loss = train_step(real_imgs, real_labels)
        d_losses.append(float(d_loss))
        g_losses.append(float(g_loss))

    d_epoch = np.mean(d_losses)
    g_epoch = np.mean(g_losses)
    d_hist.append(d_epoch)
    g_hist.append(g_epoch)

    if epoch == 1 or epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | D_loss: {d_epoch:.4f} | G_loss: {g_epoch:.4f}")
        generate_grid(epoch)

# Curvas de pérdidas
plt.figure(figsize=(8, 4))
plt.plot(d_hist, label="D_loss")
plt.plot(g_hist, label="G_loss")
plt.title("Curvas de entrenamiento GAN (D_loss vs G_loss)")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.tight_layout()
plt.show()

### 4.2.6 Análisis de diversidad (*mode collapse*)

Un problema frecuente en GANs es el ***mode collapse***: el Generator aprende a producir solo un tipo de imagen que engaña al Discriminator, perdiendo diversidad.

Verificamos generando múltiples muestras de la misma clase con ruido diferente — si las imágenes son muy similares entre sí, hay *mode collapse*.

In [ ]:
def generate_many_for_class(class_name, n=12):
    cls_id = gan_class_names.index(class_name)
    noise = tf.random.normal((n, LATENT_DIM))
    labels = tf.constant([cls_id]*n, dtype=tf.int32)

    imgs = G([noise, labels], training=False)
    imgs = (imgs.numpy() + 1.0) / 2.0

    plt.figure(figsize=(12, 3))
    for i in range(n):
        plt.subplot(2, n//2, i+1)
        plt.imshow(imgs[i])
        plt.axis("off")
    plt.suptitle(f"Imágenes generadas — clase: {class_name}")
    plt.tight_layout()
    plt.show()

for cls_name in gan_class_names:
    generate_many_for_class(cls_name, n=12)

---
## 4.3 Modelo preentrenado: Stable Diffusion

### Fundamento

**Stable Diffusion** es un modelo de difusión latente (*Latent Diffusion Model*) que genera imágenes a partir de texto (*text-to-image*). Opera en un espacio latente comprimido, lo que lo hace más eficiente que operar directamente en espacio de píxeles.

Componentes clave:
- **Text Encoder** (CLIP): Codifica el prompt en un *embedding* semántico
- **U-Net**: Red de *denoising* que genera iterativamente la imagen en espacio latente
- **VAE Decoder**: Transforma el latente en imagen de alta resolución (512×512)

Usamos el modelo `runwayml/stable-diffusion-v1-5` en modo inferencia (sin fine-tuning).

### 4.3.1 Preparación de *captions* para las categorías

Creamos prompts descriptivos para cada categoría que Stable Diffusion usará para guiar la generación.

In [ ]:
GEN_DIR = Path("gen_pretrained")
GEN_DIR.mkdir(exist_ok=True)

metadata_path = GEN_DIR / "metadata.jsonl"

def make_caption(cls):
    return f"a photo of a {cls} scene"

img_rows = []
img_exts = {".jpg", ".jpeg", ".png", ".webp"}
for cls in classes:
    for p in (DATA_DIR/cls).rglob("*"):
        if p.suffix.lower() in img_exts and p.is_file():
            img_rows.append({"file_name": str(p), "text": make_caption(cls)})

print("Total imágenes:", len(img_rows))

with open(metadata_path, "w", encoding="utf-8") as f:
    for r in img_rows:
        f.write(json.dumps(r) + "\n")

print("Guardado:", metadata_path)

### 4.3.2 Generación con prompts (baseline preentrenado)

Generamos imágenes usando prompts simples: `"a photo of a {clase} scene"`. Usamos 25 pasos de inferencia y *guidance scale* = 7.5 (equilibrio entre calidad y adherencia al prompt).

In [ ]:
MODEL_ID = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    safety_checker=None
).to("cuda" if torch.cuda.is_available() else "cpu")

pipe.set_progress_bar_config(disable=True)

def show_grid_images(images, title, cols=5):
    rows = int(np.ceil(len(images)/cols))
    plt.figure(figsize=(3*cols, 3*rows))
    for i, im in enumerate(images):
        plt.subplot(rows, cols, i+1)
        plt.imshow(im)
        plt.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

# Generar imágenes por clase
baseline_imgs = []
for cls in classes:
    prompt = make_caption(cls)
    im = pipe(prompt, num_inference_steps=25, guidance_scale=7.5).images[0]
    baseline_imgs.append(im)

show_grid_images(baseline_imgs, "Baseline SD (sin fine-tuning): 1 imagen por clase", cols=5)

### 4.3.3 Nota sobre Fine-tuning (LoRA)

Se exploró la posibilidad de realizar *fine-tuning* mediante **LoRA** (*Low-Rank Adaptation*) para adaptar Stable Diffusion al estilo visual específico de nuestro dataset. Sin embargo, las limitaciones de memoria del entorno local (sin GPU dedicada con VRAM suficiente) impidieron completar el proceso.

En un escenario con GPU adecuada (≥16 GB VRAM), LoRA permitiría adaptar el modelo preservando el conocimiento general pero aprendiendo detalles estilísticos del dataset.

---
## 4.4 Comparación cualitativa

### Criterios de evaluación

| Criterio | Descripción |
|----------|-------------|
| **Realismo** | ¿La imagen parece una fotografía real? |
| **Coherencia** | ¿La imagen corresponde a la categoría objetivo? |
| **Detalle** | Nitidez, texturas definidas, ausencia de artefactos |
| **Diversidad** | Variedad entre muestras de la misma clase |

---
## Conclusiones

### Comparación final

| Aspecto | cDCGAN (from scratch) | Stable Diffusion (preentrenado) |
|---------|----------------------|-------------------------------|
| **Realismo** | Bajo — imágenes borrosas, artefactos frecuentes | Alto — fotorrealista en la mayoría de casos |
| **Coherencia con clase** | Parcial — captura colores/texturas distintivas | Alta — sigue fielmente el prompt |
| **Diversidad** | Media — riesgo de *mode collapse* | Alta — cada generación es única |
| **Resolución** | 64×64 | 512×512 |
| **Entrenamiento** | ~100 epochs (~15-30 min) | No requerido (solo inferencia) |
| **Control** | Condicional por clase (label) | Por texto (prompt flexible) |

### Aprendizajes

1. **Las GANs desde cero son viables pero limitadas** con datasets pequeños: aprenden distribuciones básicas (color, textura) pero no composiciones semánticas complejas
2. **Los modelos de difusión preentrenados** representan el estado del arte actual en generación, con un salto cualitativo masivo respecto a las GANs clásicas
3. **El condicionalismo** funciona en ambos casos: la GAN genera paletas de color diferentes por clase, y Stable Diffusion sigue los prompts textuales
4. **Las curvas D/G loss** del entrenamiento GAN permiten monitorizar la estabilidad del entrenamiento (equilibrio entre Generator y Discriminator)